In [4]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

def parse_paths():
    if len(sys.argv) > 2 and not sys.argv[1].startswith('-f'):
        return Path(sys.argv[1]), Path(sys.argv[2])
    cands = [
        Path('./dataset/public'),
        Path('/kaggle/input/datasets/devansh66/logical-bundle'),
        Path('/kaggle/input/logical-bundle'),
        Path('/kaggle/input'),
        Path('.')
    ]
    pub = next((c for c in cands if (c / 'tasks.json').exists()), Path('./dataset/public'))
    out = Path('./working/submission.csv')
    os.makedirs(out.parent, exist_ok=True)
    return pub, out

PUB, OUT = parse_paths()
catalog = json.load(open(PUB / 'catalog.json', encoding='utf-8'))
tasks = json.load(open(PUB / 'tasks.json', encoding='utf-8'))

item_map = {item['item_id']: item for item in catalog}
item_ids = [item['item_id'] for item in catalog]
n_items = len(catalog)

cats = sorted(list(set(item['category'] for item in catalog)))
colors = sorted(list(set(item['color'] for item in catalog)))
brands = sorted(list(set(item['brand'] for item in catalog)))

cat_to_idx = {c: i for i, c in enumerate(cats)}
col_to_idx = {c: i for i, c in enumerate(colors)}
bra_to_idx = {b: i for i, b in enumerate(brands)}

prices = np.array([item['price'] for item in catalog], dtype=np.float32)
ratings = np.array([item.get('rating', 0.0) for item in catalog], dtype=np.float32)

max_p = float(np.max(prices)) if len(prices) > 0 and np.max(prices) > 0 else 1.0
max_r = float(np.max(ratings)) if len(ratings) > 0 and np.max(ratings) > 0 else 1.0

norm_prices = prices / max_p
norm_ratings = ratings / max_r

dim_cat = len(cats)
dim_col = len(colors)
dim_bra = len(brands)
total_dim = dim_cat + dim_col + dim_bra + 2

feature_matrix = np.zeros((n_items, total_dim), dtype=np.float32)
id_to_matrix_idx = {}

for idx, item in enumerate(catalog):
    id_to_matrix_idx[item['item_id']] = idx
    offset = 0
    feature_matrix[idx, offset + cat_to_idx[item['category']]] = 1.0
    offset += dim_cat
    feature_matrix[idx, offset + col_to_idx[item['color']]] = 1.0
    offset += dim_col
    feature_matrix[idx, offset + bra_to_idx[item['brand']]] = 1.0
    offset += dim_bra
    feature_matrix[idx, offset] = norm_prices[idx]
    feature_matrix[idx, offset + 1] = norm_ratings[idx]

norm_matrix = feature_matrix / (np.linalg.norm(feature_matrix, axis=1, keepdims=True) + 1e-8)

category_item_indices = defaultdict(list)
for idx, item in enumerate(catalog):
    category_item_indices[item['category']].append(idx)

def extract_slots(example_output):
    return sorted(example_output, key=lambda i: (item_map[i]['category'], i))

submission_rows = []

for task_id, task in tasks.items():
    train_examples = task['train']
    ins = [ex['input'] for ex in train_examples]
    outs = [extract_slots(ex['output']) for ex in train_examples]
    
    num_slots = len(outs[0])
    target_cats = [item_map[o]['category'] for o in outs[0]]
    
    test_input = task['test'][0]['input']
    test_indices = [id_to_matrix_idx[x] for x in test_input if x in id_to_matrix_idx]
    test_inp_vec = np.mean(norm_matrix[test_indices], axis=0) if test_indices else np.zeros(total_dim, dtype=np.float32)
    
    exclude_set = set(test_input)
    predicted_item_ids = []

    for slot_idx in range(num_slots):
        slot_cat = target_cats[slot_idx]
        slot_deltas = []
        
        for ex_idx, ex in enumerate(train_examples):
            inp_idx_list = [id_to_matrix_idx[x] for x in ins[ex_idx] if x in id_to_matrix_idx]
            out_item_id = outs[ex_idx][slot_idx]
            out_matrix_idx = id_to_matrix_idx[out_item_id]
            
            inp_mean = np.mean(norm_matrix[inp_idx_list], axis=0) if inp_idx_list else np.zeros(total_dim, dtype=np.float32)
            out_vec = norm_matrix[out_matrix_idx]
            slot_deltas.append(out_vec - inp_mean)
            
        slot_representation = np.mean(slot_deltas, axis=0)
        target_embedding = test_inp_vec + slot_representation
        target_embedding = target_embedding / (np.linalg.norm(target_embedding) + 1e-8)
        
        cand_indices = category_item_indices.get(slot_cat, [])
        valid_cands = [i for i in cand_indices if item_ids[i] not in exclude_set]
        
        if valid_cands:
            cand_matrix = norm_matrix[valid_cands]
            scores = np.dot(cand_matrix, target_embedding)
            best_local_idx = int(np.argmax(scores))
            best_cand_idx = valid_cands[best_local_idx]
            best_id = item_ids[best_cand_idx]
            predicted_item_ids.append(best_id)
            exclude_set.add(best_id)
        else:
            all_valid = [i for i in range(n_items) if item_ids[i] not in exclude_set]
            if all_valid:
                cand_matrix = norm_matrix[all_valid]
                scores = np.dot(cand_matrix, target_embedding)
                best_local_idx = int(np.argmax(scores))
                best_cand_idx = all_valid[best_local_idx]
                best_id = item_ids[best_cand_idx]
                predicted_item_ids.append(best_id)
                exclude_set.add(best_id)

    if len(predicted_item_ids) < num_slots:
        for slot_cat in target_cats:
            if len(predicted_item_ids) >= num_slots:
                break
            fillers = [item_ids[i] for i in category_item_indices.get(slot_cat, []) if item_ids[i] not in exclude_set]
            if fillers:
                predicted_item_ids.append(fillers[0])
                exclude_set.add(fillers[0])

    if not predicted_item_ids:
        predicted_item_ids = [catalog[0]['item_id']]

    submission_rows.append((task_id, ' '.join(str(x) for x in predicted_item_ids)))

sub_df = pd.DataFrame(submission_rows, columns=['task_id', 'recommended_item_ids'])
ss_df = pd.read_csv(PUB / 'sample_submission.csv')
sub_df = ss_df[['task_id']].merge(sub_df, on='task_id', how='left')
sub_df['recommended_item_ids'] = sub_df['recommended_item_ids'].fillna(str(catalog[0]['item_id']))

sub_df.to_csv(OUT, index=False)